# Build Modeling Dataset

## Objective
This notebook constructs the final loan-month modeling dataset used in all experiments.

The output dataset includes:
- loan-month observations
- next-month event labels
- macroeconomic features from TB3MS
- borrower origination features
- missingness indicators
- seasoning interaction features

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/loan_project_ncr"
os.makedirs(PROJECT_DIR, exist_ok=True)

PROSPER_PATH = os.path.join(PROJECT_DIR, "prosperLoanData.csv")
TB3MS_CANDIDATES = [
    os.path.join(PROJECT_DIR, "TB3MS.csv"),
    os.path.join(PROJECT_DIR, "TB3MS (1).csv"),
    "TB3MS.csv",
    "TB3MS (1).csv",
]
TB3MS_PATH = next((p for p in TB3MS_CANDIDATES if os.path.exists(p)), None)

if not os.path.exists(PROSPER_PATH):
    raise FileNotFoundError(f"Missing Prosper file: {PROSPER_PATH}")
if TB3MS_PATH is None:
    raise FileNotFoundError(f"Missing TB3MS file. Tried: {TB3MS_CANDIDATES}")

print("Using files:")
print("Prosper:", PROSPER_PATH)
print("TB3MS:", TB3MS_PATH)

Using files:
Prosper: /content/drive/MyDrive/loan_project_ncr/prosperLoanData.csv
TB3MS: /content/drive/MyDrive/loan_project_ncr/TB3MS.csv


## Load Source Data
We combine:
1. Prosper loan-level data
2. TB3MS monthly interest rate data

Prosper provides borrower and loan attributes at origination.
TB3MS provides a monthly rate series used to construct refinancing incentive features.

In [3]:
df = pd.read_csv(PROSPER_PATH)

df["ListingCreationDate"] = pd.to_datetime(df["ListingCreationDate"], errors="coerce")
df["ClosedDate"] = pd.to_datetime(df.get("ClosedDate"), errors="coerce")

df = df.dropna(subset=["ListingCreationDate", "BorrowerRate"]).copy()

# Safe origination-time categorical encoding
df["IncomeRange"] = df["IncomeRange"].astype("category").cat.codes
df["EmploymentStatus"] = df["EmploymentStatus"].astype("category").cat.codes
df["IsBorrowerHomeowner"] = df.get("IsBorrowerHomeowner", False)
df["IsBorrowerHomeowner"] = df["IsBorrowerHomeowner"].fillna(False).astype(int)

print(df.shape)
df.head()

(113550, 81)


,ListingKey,ListingNumber,ListingCreationDate,CreditGrade,Term,LoanStatus,ClosedDate,BorrowerAPR,BorrowerRate,LenderYield,...,LP_ServiceFees,LP_CollectionFees,LP_GrossPrincipalLoss,LP_NetPrincipalLoss,LP_NonPrincipalRecoverypayments,PercentFunded,Recommendations,InvestmentFromFriendsCount,InvestmentFromFriendsAmount,Investors
0,1021339766868145413AB3B,193129,2007-08-26 19:09:29.263,C,36,Completed,2009-08-14,0.16516,0.1580,0.1380,...,-133.18,0.0,0.0,0.0,0.0,1.0,0,0,0.0,258
1,10273602499503308B223C1,1209647,2014-02-27 08:28:07.900,NaN,36,Current,NaT,0.12016,0.0920,0.0820,...,0.00,0.0,0.0,0.0,0.0,1.0,0,0,0.0,1
2,0EE9337825851032864889A,81716,2007-01-05 15:00:47.090,HR,36,Completed,2009-12-17,0.28269,0.2750,0.2400,...,-24.20,0.0,0.0,0.0,0.0,1.0,0,0,0.0,41
3,0EF5356002482715299901A,658116,2012-10-22 11:02:35.010,NaN,36,Current,NaT,0.12528,0.0974,0.0874,...,-108.01,0.0,0.0,0.0,0.0,1.0,0,0,0.0,158
4,0F023589499656230C5E3E2,909464,2013-09-14 18:38:39.097,NaN,36,Current,NaT,0.24614,0.2085,0.1985,...,-60.27,0.0,0.0,0.0,0.0,1.0,0,0,0.0,20


## Event Label Construction

We expand each loan into monthly observations and define:

- `0 = continue`
- `1 = prepay`
- `2 = default`

The resulting event label is stored in `status_next`.

In [4]:
NEED_LABELS = (("LastObservedMonth" not in df.columns) or ("Month_1" not in df.columns))

if NEED_LABELS:
    print("Creating LastObservedMonth and Month_1..Month_36 labels.")

    AS_OF_DATE = pd.Timestamp("2014-03-10")

    if "Term" not in df.columns:
        df["Term"] = 36

    loan_status = df["LoanStatus"].astype(str)
    is_completed = loan_status.isin(["Completed", "FinalPaymentInProgress"])
    is_default = loan_status.isin(["Defaulted", "Chargedoff"])

    months_to_close = (
        df["ClosedDate"].dt.to_period("M") - df["ListingCreationDate"].dt.to_period("M")
    ).apply(lambda x: x.n if pd.notna(x) else np.nan)

    months_to_asof = (
        AS_OF_DATE.to_period("M") - df["ListingCreationDate"].dt.to_period("M")
    ).apply(lambda x: x.n if pd.notna(x) else np.nan)

    is_prepay = is_completed & months_to_close.notna() & (months_to_close < df["Term"])

    months_event_or_censor = np.where(
        (is_default | is_completed) & months_to_close.notna(),
        months_to_close,
        months_to_asof
    )

    months_event_or_censor = pd.Series(months_event_or_censor).clip(lower=0)
    df["LastObservedMonth"] = np.minimum(
        36, months_event_or_censor.round().fillna(0).astype(int)
    )

    for m in range(1, 37):
        df[f"Month_{m}"] = np.where(df["LastObservedMonth"] >= m, 0.0, np.nan)

    event_month = months_to_close.round()

    default_mask = is_default & event_month.notna() & (event_month >= 1) & (event_month <= 36)
    prepay_mask = is_prepay & event_month.notna() & (event_month >= 1) & (event_month <= 36)

    for m in range(1, 37):
        df.loc[default_mask & (event_month == m), f"Month_{m}"] = 2.0
        df.loc[prepay_mask & (event_month == m), f"Month_{m}"] = 1.0
else:
    print("Month labels already found. Skipping label creation.")

Creating LastObservedMonth and Month_1..Month_36 labels.


## Load TB3MS and Build Monthly Rate Table
TB3MS is resampled to month start so that each loan-month can be aligned with a monthly market rate.

In [5]:
tb = pd.read_csv(TB3MS_PATH)

date_col = "DATE" if "DATE" in tb.columns else "observation_date"
rate_col = "TB3MS" if "TB3MS" in tb.columns else [c for c in tb.columns if c != date_col][0]

tb[date_col] = pd.to_datetime(tb[date_col], errors="coerce")
tb = tb.dropna(subset=[date_col]).sort_values(date_col).drop_duplicates(date_col)

tb_monthly = (
    tb.set_index(date_col)
      .resample("MS")
      .interpolate("linear")
)

tb_dict = tb_monthly[rate_col].to_dict()
print(f"TB3MS months available: {len(tb_dict)}")

TB3MS months available: 301


## Build Loan-Month Dataset

For each loan-month we compute:
- month since origination
- current TB3MS level
- 3-month TB3MS change
- APR spread features
- borrower origination features
- next-month event label

In [6]:
monthly_rows = []
skipped_apr = 0

for loan_id, loan in enumerate(df.itertuples(index=False)):
    borrower_apr = getattr(loan, "BorrowerRate")
    if borrower_apr is None or np.isnan(borrower_apr) or borrower_apr <= 0:
        skipped_apr += 1
        continue

    last_month = getattr(loan, "LastObservedMonth", None)
    if last_month is None or pd.isna(last_month):
        continue

    last_month = int(last_month)
    if last_month <= 0:
        continue

    orig_date = getattr(loan, "ListingCreationDate")
    if pd.isna(orig_date):
        continue

    loan_amt = getattr(loan, "LoanOriginalAmount", 10000.0)
    if loan_amt is None or pd.isna(loan_amt):
        loan_amt = 10000.0

    max_m = min(36, last_month)

    for m in range(1, max_m + 1):
        month_date = (orig_date + pd.DateOffset(months=m - 1)).to_period("M").to_timestamp()
        month_date_3mo = (month_date - pd.DateOffset(months=3)).to_period("M").to_timestamp()

        tb_now = tb_dict.get(month_date, np.nan)
        if pd.isna(tb_now):
            tb_now = 0.50

        tb_3mo = tb_dict.get(month_date_3mo, tb_now)

        tb_now_dec = tb_now / 100.0
        tb_3mo_dec = tb_3mo / 100.0

        apr_tbill_spread = borrower_apr - tb_now_dec
        tbill_change_3m = tb_now_dec - tb_3mo_dec
        apr_tbill_spread_change_3m = (borrower_apr - tb_3mo_dec) - apr_tbill_spread

        status_next = getattr(loan, f"Month_{m}")

        monthly_rows.append({
            "loan_id": loan_id,
            "cohort_year": orig_date.year,
            "month_since_orig": m,
            "status_next": status_next,
            "tbill_level": tb_now_dec,
            "tbill_change_3m": tbill_change_3m,
            "apr_tbill_spread": apr_tbill_spread,
            "apr_tbill_spread_change_3m": apr_tbill_spread_change_3m,
            "apr_tbill_spread_pct": apr_tbill_spread / borrower_apr,
            "borrower_apr": borrower_apr,
            "loan_amount_log": np.log1p(loan_amt),
            "prosper_score": getattr(loan, "ProsperScore", np.nan),
            "credit_score_lower": getattr(loan, "CreditScoreRangeLower", np.nan),
            "credit_score_upper": getattr(loan, "CreditScoreRangeUpper", np.nan),
            "dti": getattr(loan, "DebtToIncomeRatio", np.nan),
            "delinq_7y": getattr(loan, "DelinquenciesLast7Years", np.nan),
            "open_credit_lines": getattr(loan, "OpenCreditLines", np.nan),
            "revolving_balance": getattr(loan, "RevolvingCreditBalance", np.nan),
            "bankcard_util": getattr(loan, "BankcardUtilization", np.nan),
            "available_bankcard_credit": getattr(loan, "AvailableBankcardCredit", np.nan),
            "income_range": getattr(loan, "IncomeRange", np.nan),
            "employment_status": getattr(loan, "EmploymentStatus", np.nan),
            "homeowner": getattr(loan, "IsBorrowerHomeowner", np.nan),
        })

monthly_df = pd.DataFrame(monthly_rows)

print("monthly_df shape:", monthly_df.shape)
print("Skipped loans (APR <= 0):", skipped_apr)
print(monthly_df["status_next"].value_counts(dropna=False).sort_index())
monthly_df.head()

monthly_df shape: (1656440, 23)
Skipped loans (APR <= 0): 7
status_next
0.0    1636619
1.0      12934
2.0       6887
Name: count, dtype: int64


,loan_id,cohort_year,month_since_orig,status_next,tbill_level,tbill_change_3m,apr_tbill_spread,apr_tbill_spread_change_3m,apr_tbill_spread_pct,borrower_apr,...,credit_score_upper,dti,delinq_7y,open_credit_lines,revolving_balance,bankcard_util,available_bankcard_credit,income_range,employment_status,homeowner
0,0,2007,1,0.0,0.0420,-0.0053,0.1160,-0.0053,0.734177,0.158,...,659.0,0.17,4.0,4.0,0.0,0.0,1500.0,3,7,1
1,0,2007,2,0.0,0.0389,-0.0072,0.1191,-0.0072,0.753797,0.158,...,659.0,0.17,4.0,4.0,0.0,0.0,1500.0,3,7,1
2,0,2007,3,0.0,0.0390,-0.0092,0.1190,-0.0092,0.753165,0.158,...,659.0,0.17,4.0,4.0,0.0,0.0,1500.0,3,7,1
3,0,2007,4,0.0,0.0327,-0.0093,0.1253,-0.0093,0.793038,0.158,...,659.0,0.17,4.0,4.0,0.0,0.0,1500.0,3,7,1
4,0,2007,5,0.0,0.0300,-0.0089,0.1280,-0.0089,0.810127,0.158,...,659.0,0.17,4.0,4.0,0.0,0.0,1500.0,3,7,1


## Add Missingness Indicators
For borrower variables with missing values, we create indicator columns so that downstream models can use missingness as signal.

In [7]:
borrower_cols = [
    "prosper_score", "credit_score_lower", "credit_score_upper", "dti", "delinq_7y",
    "open_credit_lines", "revolving_balance", "bankcard_util", "available_bankcard_credit",
    "income_range", "employment_status", "homeowner"
]

for c in borrower_cols:
    monthly_df[c + "_is_missing"] = monthly_df[c].isna().astype(np.int8)

monthly_df[[c + "_is_missing" for c in borrower_cols]].head()

,prosper_score_is_missing,credit_score_lower_is_missing,credit_score_upper_is_missing,dti_is_missing,delinq_7y_is_missing,open_credit_lines_is_missing,revolving_balance_is_missing,bankcard_util_is_missing,available_bankcard_credit_is_missing,income_range_is_missing,employment_status_is_missing,homeowner_is_missing
0,1,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0


## Add Seasoning Interaction Features
These features capture the interaction between refinancing incentive and loan age.

In [8]:
monthly_df["spread_x_seasoning"] = (
    monthly_df["apr_tbill_spread"] * monthly_df["month_since_orig"]
)
monthly_df["rate_change_x_seasoning"] = (
    monthly_df["tbill_change_3m"] * monthly_df["month_since_orig"]
)
monthly_df["spread_pct_x_seasoning"] = (
    monthly_df["apr_tbill_spread_pct"] * monthly_df["month_since_orig"]
)

monthly_df[
    ["spread_x_seasoning", "rate_change_x_seasoning", "spread_pct_x_seasoning"]
].describe()

,spread_x_seasoning,rate_change_x_seasoning,spread_pct_x_seasoning
count,1.656440e+06,1.656440e+06,1.656440e+06
mean,2.294619e+00,-8.598273e-03,1.184344e+01
std,2.087538e+00,4.136430e-02,9.186749e+00
min,-2.838000e-01,-5.355000e-01,-5.130000e+02
25%,7.085000e-01,-2.800000e-03,3.993180e+00
50%,1.654100e+00,-5.000000e-04,9.837104e+00
75%,3.243900e+00,9.000000e-04,1.795332e+01
max,1.609200e+01,1.920000e-01,3.598971e+01


## Save Final Modeling Dataset

This parquet file will be used by all modeling notebooks.

In [9]:
ENGINEERED_PATH = os.path.join(
    PROJECT_DIR,
    "monthly_df_tb3ms_borrower_seasoning.parquet"
)

monthly_df.to_parquet(ENGINEERED_PATH, index=False)

print("Saved engineered dataset to:")
print(ENGINEERED_PATH)
print("Final shape:", monthly_df.shape)

Saved engineered dataset to:
/content/drive/MyDrive/loan_project_ncr/monthly_df_tb3ms_borrower_seasoning.parquet
Final shape: (1656440, 38)
